In [1]:
!pip install vllm transformers accelerate

In [2]:
!pip install litellm

In [3]:
!nohup vllm serve Qwen/Qwen3-1.7B \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --reasoning-parser deepseek_r1 \
  > vllm.log &

nohup: redirecting stderr to stdout


In [12]:
!tail -n 20 vllm.log # to watch last 20 rows og vllm.log, about 2/3 minutes to have the server ready -> Application startup complete

(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/chat/completions, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/completions, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/embeddings, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /pooling, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /classify, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /score, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/score, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=63241) INFO 11-11 17:52:07 [launcher.py:42] Route: /v1/audio/translations, Methods: POST
(APIServer pid=6324

In [13]:
import sqlite3
import pandas as pd

def query_db(sql: str):
    """
    Executes an SQL query and returns the result as JSON

    Args:
        sql: The SQL query to execute

    Returns:
        The result of the query as JSON
    """
    try:
        df = pd.read_sql_query(sql, conn)
        return df.to_dict(orient="records")
    except Exception as e:
        return {"error": str(e)}

In [55]:
db_name = "shakespeare.sqlite"
conn = sqlite3.connect(db_name)

In [56]:
def get_schema(connection):
    schema_str = ""
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table_name in tables:
        table_name = table_name[0]
        if table_name.startswith("sqlite_"):
            continue

        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_statement = cursor.fetchone()[0]
        schema_str += create_statement + ";\n"

    return schema_str

In [57]:
db_schema = get_schema(conn)

In [58]:
def get_function_by_name(name):
    if name == "query_db":
        return query_db

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_db",
            "description": "Executes a single SQL query on the database. Use this to make exploratory queries in order to find the final answer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": 'The SQL query to execute on the database.',
                    }
                },
                "required": ["sql"],
            },
        },
    }
]

In [59]:
USER_QUESTION = f"""
Database Schema:
{db_schema}
Evidence:
King John refers to Title = 'King John'
Question:
How many scenes are there in King John?.
"""



SYSTEM_PROMPT = """You are an expert SQL assistant. Your task is to answer the user question by generating a final SQL query.
The schema of the database is provided in the user question.
You can use the evidence provided in the user question to ensure that ambiguous natural language terms are correctly translated into the specific SQL logic required by the database schema.

To assist your reasoning, you MUST use the `query_db` tool to run exploratory "probe" queries.
These probes are essential for understanding the data types, relationships, and potential values.

You MUST follow a strict reasoning process using specific tags in your thought process.
Your reasoning MUST be formatted using these tags:

[GOAL] (once) At the beginning, restate the question in your own words. What exactly must be returned? State if you are making assumptions.
[STEP] (possibly multiple) Your current plan (high level). E.g., which tables are involved or which joins do you expect? Which filters/aggregations? Keep it brief but explicit.
If the step is after a tool call, state your concise observation from the `[ANS]` result (e.g., "The tables 'employees' and 'departments' are confirmed", "status=1 seems correct"). This confirms or refutes your hypothesis.
[FINAL] (once) At the end, a brief statement on the conclusion of the process before giving the final result query.

How the tool calling works:

1. After your [STEP] (for a probe query) you MUST call the `query_db` tool with an exploratory query. This query MUST include `LIMIT 3`.
If the model produces a SQL query without `LIMIT 3` before the [FINAL] step, it is INVALID and must be corrected immediately.
2. The main script (not you) will then print a [CALL] tag followed by your SQL query.
3. After executing the query, the script will print an [ANS] tag with the results (up to 3 rows for probes, or "(no rows)" if empty).
4. You will see this [ANS] result in the chat history. You MUST use this result to inform your next [STEP].
5. After the [FINAL] reasoning tag, you MUST NOT call any tool. Instead, you MUST output the final, complete query string (without `LIMIT 3`, unless needed for the query purposes).

"""

In [60]:
from litellm import completion
import json

model_name = "hosted_vllm/Qwen/Qwen3-1.7B"
MAX_TURNS = 10 # a maximum number of turns in order to avoid infinite loops

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION}
]

completion_params = {
    "model": model_name,
    "temperature": 0.1,
    "top_p": 0.8,
    "max_tokens": 1024,
    "repetition_penalty": 1.05,

    "extra_body": {
        "chat_template_kwargs": {
            "enable_thinking": True,
            "detailed_thinking": True
        }
    },
    "api_base": "http://127.0.0.1:8000/v1",
    "tools": TOOLS
}

print(USER_QUESTION)

for i in range(MAX_TURNS):

    # call the model
    response = completion(messages=messages, **completion_params)
    response_message = response.choices[0].message

    # add the model's response (thought + tool_call) to the history
    messages.append(response_message.model_dump())

    # print of the model's reasoning trace
    if response_message.content:
         print(f"{response_message.content.strip()}")

    # check if the model has called a tool
    if tool_calls := response_message.get("tool_calls", None):

        # execution of the tool call
        for tool_call in tool_calls:
            call_id = tool_call["id"]
            if fn_call := tool_call.get("function"):
                fn_name = fn_call["name"]
                fn_args = json.loads(fn_call["arguments"])
                sql_query = fn_args.get("sql", "")

                # print [CALL] in the requested format
                print(f"\n[CALL]\n{sql_query.strip()};")

                # Execute the function
                fn_res = get_function_by_name(fn_name)(**fn_args)

                # Format and print [ANS] in the requested format
                result_str = ""
                if isinstance(fn_res, list) and len(fn_res) > 0:
                    # Convert list of dicts back to DataFrame for nice printing
                    df_res = pd.DataFrame(fn_res)
                    result_str = df_res.to_string(index=False)
                elif isinstance(fn_res, list) and len(fn_res) == 0:
                    result_str = "(no rows)"
                else:
                    result_str = str(fn_res)

                print(f"[ANS]\n{result_str}\n[/ANS]")


                #add the tool result (as JSON) to the history for the model
                messages.append({
                    "role": "tool",
                    "content": json.dumps({
                        "query_executed": sql_query,
                        "result": fn_res
                    }),
                    "tool_call_id": call_id,
                })

    # Check if the model is finished (gave a final answer without tool calls)
    elif response_message.content:
        break


conn.close()


Database Schema:
CREATE TABLE "chapters"
(
    id          INTEGER
        primary key autoincrement,
    Act         INTEGER not null,
    Scene       INTEGER not null,
    Description TEXT    not null,
    work_id     INTEGER not null
        references works
);
CREATE TABLE "characters"
(
    id          INTEGER
        primary key autoincrement,
    CharName    TEXT not null,
    Abbrev      TEXT not null,
    Description TEXT not null
);
CREATE TABLE "paragraphs"
(
    id           INTEGER
        primary key autoincrement,
    ParagraphNum INTEGER           not null,
    PlainText    TEXT              not null,
    character_id INTEGER           not null
        references characters,
    chapter_id   INTEGER default 0 not null
        references chapters
);
CREATE TABLE "works"
(
    id        INTEGER
        primary key autoincrement,
    Title     TEXT    not null,
    LongTitle TEXT    not null,
    Date      INTEGER not null,
    GenreType TEXT    not null
);

Evidence:
Kin

In [61]:
messages

[{'role': 'system',
  'content': 'You are an expert SQL assistant. Your task is to answer the user question by generating a final SQL query.\nThe schema of the database is provided in the user question.\nYou can use the evidence provided in the user question to ensure that ambiguous natural language terms are correctly translated into the specific SQL logic required by the database schema.\n\nTo assist your reasoning, you MUST use the `query_db` tool to run exploratory "probe" queries.\nThese probes are essential for understanding the data types, relationships, and potential values.\n\nYou MUST follow a strict reasoning process using specific tags in your thought process.\nYour reasoning MUST be formatted using these tags:\n\n[GOAL] (once) At the beginning, restate the question in your own words. What exactly must be returned? State if you are making assumptions.\n[STEP] (possibly multiple) Your current plan (high level). E.g., which tables are involved or which joins do you expect? Wh